# DEH 30-Day PySpark Challenge
## Day 12 — Date Functions

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name",  StringType(), True),
    StructField("last_name",   StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("state",       StringType(), True),
    StructField("country",     StringType(), True),
    StructField("signup_date", DateType(),   True),
    StructField("segment",     StringType(), True)
])

orders_df    = spark.read.option("header", "true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header", "true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")

print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()}")

## Step 5 — current_date() and current_timestamp()

In [ ]:
spark.range(1).select(
    F.current_date().alias("today"),
    F.current_timestamp().alias("now")
).show(truncate=False)

## Step 6 — Extracting date parts

In [ ]:
orders_df.select(
    F.col("order_date"),
    F.year(F.col("order_date")).alias("year"),
    F.month(F.col("order_date")).alias("month"),
    F.dayofmonth(F.col("order_date")).alias("day"),
    F.dayofweek(F.col("order_date")).alias("day_of_week"),
    F.quarter(F.col("order_date")).alias("quarter")
).show(5)

## Step 7 — date_add(), date_sub(), datediff()

In [ ]:
orders_df.select(
    F.col("order_date"),
    F.date_add(F.col("order_date"),  7).alias("plus_7_days"),
    F.date_sub(F.col("order_date"), 30).alias("minus_30_days"),
    F.datediff(F.current_date(), F.col("order_date")).alias("days_since_order")
).show(3)

## Step 8 — to_date() and date_format()

In [ ]:
# to_date — parse string to DateType
date_df = spark.createDataFrame(
    [("2023-01-05",), ("2023-02-15",), ("2023-03-22",)],
    ["date_str"]
)

date_df.select(
    F.col("date_str"),
    F.to_date(F.col("date_str"), "yyyy-MM-dd").alias("date_type")
).show()

In [ ]:
# date_format — format date as string for output
orders_df.select(
    F.col("order_date"),
    F.date_format(F.col("order_date"), "dd/MM/yyyy").alias("uk_format"),
    F.date_format(F.col("order_date"), "MMM dd, yyyy").alias("readable"),
    F.date_format(F.col("order_date"), "yyyy-MM").alias("year_month"),
    F.date_format(F.col("order_date"), "EEEE").alias("day_name")
).show(3)

---
## Assignment — Day 12

---

### Task 1
From `orders.csv`, extract `year`, `month`, `day`, and `quarter` from `order_date`.  
How many orders were placed in Q1 (quarter = 1)?

In [ ]:
# Task 1 — Write your code here


### Task 2
Calculate how many days ago each order was placed using `datediff(current_date(), order_date)`.  
Show the 5 most recent orders.

In [ ]:
# Task 2 — Write your code here


### Task 3
Add a `delivery_deadline` column — 7 days after `order_date` using `date_add()`.  
Also add a `year_month` column formatted as `yyyy-MM` using `date_format()`.

In [ ]:
# Task 3 — Write your code here


### Task 4
From `customers.csv`, use `datediff()` to calculate how many days each customer has been a member since `signup_date`.  
Show the top 5 longest-standing customers.

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*